# Rhino LLM — Chat, Merge & GGUF Export

Load **Qwen2.5-Coder-3B** + Rhino LoRA, chat in Colab, optionally merge weights, export **GGUF Q4_K_M**.

In [ ]:
!nvidia-smi

In [ ]:
%%capture
import os
os.environ["UNSLOTH_IS_COLAB"] = "1"
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q gradio transformers accelerate pyyaml

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
from getpass import getpass

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("HF token: ")
os.environ["HF_TOKEN"] = HF_TOKEN
!huggingface-cli login --token $HF_TOKEN

In [ ]:
import yaml
from pathlib import Path

REPO_DIR = Path("/content/My-LLM")
cfg_path = REPO_DIR / "configs/train_qlora.yaml" if REPO_DIR.exists() else Path("configs/train_qlora.yaml")
cfg = yaml.safe_load(cfg_path.read_text()) if cfg_path.exists() else {
    "model": {"base": "Qwen/Qwen2.5-Coder-3B-Instruct"},
    "export": {"adapter_dir": "rhino-qwen3b-lora", "merged_dir": "rhino-qwen3b-merged", "gguf_quant": "q4_k_m"},
    "system_prompt": "You are a Rhino 3D expert. Prefer Rhino.Python or RhinoCommon C# as asked. Give runnable code and brief notes.",
}

BASE_MODEL = cfg["model"]["base"]
ADAPTER_DIR = "/content/drive/MyDrive/rhino_llm/" + cfg["export"]["adapter_dir"]
MERGED_DIR = "/content/drive/MyDrive/rhino_llm/" + cfg["export"]["merged_dir"]
SYSTEM_PROMPT = cfg["system_prompt"].strip()
GGUF_QUANT = cfg["export"]["gguf_quant"]

print("Base:", BASE_MODEL)
print("Adapter:", ADAPTER_DIR)

In [ ]:
from unsloth import FastLanguageModel
from pathlib import Path

# Load adapter folder saved by 02_train (contains adapter_config.json)
adapter_path = Path(ADAPTER_DIR)
load_name = str(adapter_path) if (adapter_path / "adapter_config.json").exists() else BASE_MODEL

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=load_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
if load_name == BASE_MODEL and adapter_path.exists():
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, ADAPTER_DIR)
FastLanguageModel.for_inference(model)
print("Loaded:", load_name)

## Gradio chat

In [ ]:
import gradio as gr

def chat_fn(message, history):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for user, assistant in history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": assistant})
    messages.append({"role": "user", "content": message})
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(input_ids=inputs, max_new_tokens=768, temperature=0.3)
    reply = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
    return reply

demo = gr.ChatInterface(chat_fn, title="Rhino LLM", description="Rhino.Python / RhinoCommon assistant")
demo.launch(share=True)

## Merge LoRA (optional full HF weights)

In [ ]:
from pathlib import Path

DO_MERGE = False  # set True to write merged 16-bit weights (needs more RAM)

if DO_MERGE:
    model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
    print("Merged model saved to", MERGED_DIR)
else:
    print("Skipping merge. Set DO_MERGE=True to export full HF folder.")

## Export GGUF (Q4_K_M)

For **≤8 GB** VRAM: use Q4_K_M in Ollama / LM Studio / llama.cpp.  
For **~4 GB**: use 1.5B base + same quant.

In [ ]:
from pathlib import Path

GGUF_DIR = "/content/drive/MyDrive/rhino_llm/gguf"
Path(GGUF_DIR).mkdir(parents=True, exist_ok=True)

DO_GGUF = True  # set False to skip (can take several minutes)

if DO_GGUF:
    model.save_pretrained_gguf(
        GGUF_DIR,
        tokenizer,
        quantization_method=GGUF_QUANT,
    )
    print("GGUF export done:", GGUF_DIR)
else:
    print("Skipped GGUF export")

## Local inference notes

- **Ollama:** `ollama create rhino -f Modelfile` pointing at the `.gguf`
- **llama.cpp:** `./llama-cli -m rhino-q4_k_m.gguf -p "In Rhino.Python..."`
- **8 GB GPU:** Q4_K_M or 4-bit HF merged
- **4 GB GPU:** Qwen2.5-Coder-1.5B + Q4_K_M